# 02 — Random LoRA SFT on Korean Olympiad TDCS 3K
Train on `Seungjun/Korean-Olympiad-TDCS-3K`, then evaluate on `ChuGyouk/OlympiadBench-Math-Ko` with the same symbolic-equivalence scorer as Notebook 1.

In [ ]:
REPO_URL = "https://github.com/seungjun-green/Korean-TDCS"
!git clone {REPO_URL} korean-math-tdcs
%cd korean-math-tdcs
# Colab's preinstalled torchao 0.10 conflicts with PEFT and is not used here.
!pip uninstall -y torchao
!pip install -e .

In [ ]:
import shutil
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

drive_results = Path("/content/drive/MyDrive/Korean-TDCS/results")
local_results = Path.cwd() / "results"
drive_results.mkdir(parents=True, exist_ok=True)

if local_results.is_symlink():
    if local_results.resolve() != drive_results.resolve():
        raise RuntimeError(f"{local_results} points to the wrong Drive directory")
elif local_results.exists():
    shutil.copytree(local_results, drive_results, dirs_exist_ok=True)
    shutil.rmtree(local_results)

if not local_results.exists():
    local_results.symlink_to(drive_results, target_is_directory=True)

print(f"Saving all outputs to {drive_results}")

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))

In [ ]:
# ---- User controls ----
TRAINING_BATCH_SIZE = 16  # GPU micro-batch. Reduce to 8 only if this OOMs.
EFFECTIVE_BATCH_SIZE = 32  # Kept constant through gradient accumulation.
TRAINING_MAX_TOKENS = 3072  # Covers 99.94% of serialized examples.
TRAINING_EPOCHS = 4  # Maximum; early stopping may finish sooner.
VALIDATION_CHECKS_PER_EPOCH = 4
EARLY_STOPPING_PATIENCE = 4  # One epoch without validation-loss improvement.
EARLY_STOPPING_MIN_DELTA = 0.0
EVAL_BATCH_SIZE = 16
EVAL_MAX_TOKENS = 4096  # Maximum newly generated tokens.
SFT_RUN_DIR = Path("results/sft/olympiad_run_002")

if not 1 <= TRAINING_BATCH_SIZE <= EFFECTIVE_BATCH_SIZE:
    raise ValueError("TRAINING_BATCH_SIZE must be between 1 and EFFECTIVE_BATCH_SIZE")
if EFFECTIVE_BATCH_SIZE % TRAINING_BATCH_SIZE != 0:
    raise ValueError("EFFECTIVE_BATCH_SIZE must be divisible by TRAINING_BATCH_SIZE")
if TRAINING_EPOCHS < 1:
    raise ValueError("TRAINING_EPOCHS must be positive")
if VALIDATION_CHECKS_PER_EPOCH < 1 or EARLY_STOPPING_PATIENCE < 1:
    raise ValueError("Validation frequency and early-stopping patience must be positive")

In [ ]:
from datasets import load_dataset
from korean_math_tdcs.utils.config import load_config

preflight_config = load_config("configs/sft.yaml")
expected_dataset = "Seungjun/Korean-Olympiad-TDCS-3K"
if preflight_config["data"]["dataset"] != expected_dataset:
    raise RuntimeError(f"Wrong training dataset: {preflight_config['data']['dataset']}")
if preflight_config["data"].get("format") != "olympiad_tdcs":
    raise RuntimeError("configs/sft.yaml is not configured for olympiad_tdcs")
raw_dataset = load_dataset(expected_dataset, token=userdata.get("HF_TOKEN"))
if len(raw_dataset["train"]) != 3000 or len(raw_dataset["validation"]) != 300:
    raise RuntimeError(
        f"Unexpected split sizes: train={len(raw_dataset['train'])}, "
        f"validation={len(raw_dataset['validation'])}"
    )
required_columns = {"problem_ko", "solution_ko", "difficulty_level"}
if not required_columns.issubset(raw_dataset["train"].column_names):
    raise RuntimeError("Training dataset schema is incorrect")
planned_steps = (len(raw_dataset["train"]) * TRAINING_EPOCHS + EFFECTIVE_BATCH_SIZE - 1) // EFFECTIVE_BATCH_SIZE
if planned_steps != 375:
    raise RuntimeError(f"Expected 375 optimizer steps, resolved {planned_steps}")
print("Preflight passed: correct dataset, 3,000/300 rows, 375 optimizer steps")

audit_cmd = ("python scripts/analyze_difficulty.py --config configs/sft.yaml "
             f"--set training.max_seq_length={TRAINING_MAX_TOKENS}")
!{audit_cmd}

train_cmd = ("python scripts/train_sft.py --config configs/sft.yaml "
             f"--set training.micro_batch_size={TRAINING_BATCH_SIZE} "
             f"--set training.effective_batch_size={EFFECTIVE_BATCH_SIZE} "
             f"--set training.max_seq_length={TRAINING_MAX_TOKENS} "
             f"--set training.epochs={TRAINING_EPOCHS} "
             f"--set training.validation_checks_per_epoch={VALIDATION_CHECKS_PER_EPOCH} "
             f"--set training.early_stopping_patience={EARLY_STOPPING_PATIENCE} "
             f"--set training.early_stopping_min_delta={EARLY_STOPPING_MIN_DELTA} "
             f"--set output.run_dir={SFT_RUN_DIR}")
!{train_cmd}

In [ ]:
BASELINE_RESULTS_PATH = (
    "results/baseline/olympiad_bench_math_ko/"
    f"max_tokens_{EVAL_MAX_TOKENS}/metrics.json"
)
SFT_RESULTS_PATH = SFT_RUN_DIR / f"evaluation_max_tokens_{EVAL_MAX_TOKENS}.json"
SFT_ADAPTER_PATH = SFT_RUN_DIR / "adapter"
if not (SFT_ADAPTER_PATH / "adapter_config.json").exists():
    raise FileNotFoundError("SFT adapter missing; finish the training cell first")

cmd = ("python scripts/evaluate.py --config configs/baseline.yaml "
       f"--set evaluation.batch_size={EVAL_BATCH_SIZE} "
       f"--set evaluation.generation.max_new_tokens={EVAL_MAX_TOKENS} "
       f"--set model.adapter={SFT_ADAPTER_PATH} "
       f"--set output.results_path={SFT_RESULTS_PATH}")
!{cmd}

In [ ]:
import json
from pathlib import Path

import pandas as pd

base_path = Path(BASELINE_RESULTS_PATH)
base = json.load(base_path.open()) if base_path.exists() else None
sft = json.load(open(SFT_RESULTS_PATH))
table = {'Random SFT': {k: v['score'] for k, v in sft['benchmarks'].items()}}
if base:
    table['Base'] = {k: v['score'] for k, v in base['benchmarks'].items()}
pd.DataFrame(table)